# 02 - Pré-processamento do Dataset

Aplica um método de pré-processamento a todas as imagens do dataset 140k e salva em uma pasta dedicada para uso nos notebooks de treinamento.

Métodos disponíveis (escolha um configurando `METHOD` e `VALUE`):

| METHOD | VALUE | Descrição |
|---|---|---|
| `jpeg` | qualidade (ex: 50) | Compressão JPEG |
| `blur` | raio (ex: 3) | Gaussian blur |
| `noise` | desvio padrão (ex: 0.05) | Ruído gaussiano |
| `downscale` | fator (ex: 8) | Downscale + upscale |

> Baseado no gridsearch do notebook `01d`, os mais eficazes foram: `downscale` x8, `blur` r3, `blur` r5.

> ⚠️ **Papel deste notebook após as fases 01d–01f.** A receita escolhida treina em imagens **RAW + augmentation aleatória `jpeg+noise`** (aplicada *por época*, no `train_transform` dos notebooks 03/04). Este notebook implementa uma abordagem **alternativa**: uma degradação *fixa*, aplicada uma única vez ao dataset inteiro. As duas não se misturam — degradação estática não é "augmentation" (não há aleatoriedade por época). Por isso 03/04 usam `USE_RAW=True` por padrão e **não** consomem a saída daqui; para rodar a variante degradada (ablação), gere um dataset aqui e troque `USE_RAW=False` em 03/04. As melhorias de velocidade/split do ArtiFact vivem em 03/04 (este notebook não treina nem avalia).

In [ ]:
import io
import json
import shutil
from pathlib import Path

import numpy as np
from PIL import Image, ImageFilter
from tqdm import tqdm

In [ ]:
PROJECT_ROOT    = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT       = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"
SRC_DIR         = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"

# ── configuração ──────────────────────────────────────────────
METHOD = "downscale"   # jpeg | blur | noise | downscale
VALUE  = 8             # significado depende do método
# ─────────────────────────────────────────────────────────────

DST_DIR = DATA_ROOT / "processed" / f"140k_{METHOD}_{VALUE}"

splits  = ["train", "valid", "test"]
classes = ["real", "fake"]

print(f"Método: {METHOD} | Valor: {VALUE}")
print(f"Fonte:  {SRC_DIR}")
print(f"Destino: {DST_DIR}")

## 1. Função de Pré-processamento

In [ ]:
def preprocess(img: Image.Image, method: str, value) -> Image.Image:
    if method == "jpeg":
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=int(value))
        buf.seek(0)
        return Image.open(buf).copy()
    elif method == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=float(value)))
    elif method == "noise":
        arr = np.array(img, dtype=np.float32)
        noisy = arr + np.random.normal(0, float(value) * 255, arr.shape)
        return Image.fromarray(np.clip(noisy, 0, 255).astype(np.uint8))
    elif method == "downscale":
        factor = int(value)
        w, h = img.size
        small = img.resize((w // factor, h // factor), Image.BILINEAR)
        return small.resize((w, h), Image.BILINEAR)
    else:
        raise ValueError(f"Método desconhecido: {method}")

print("Função definida.")

## 2. Processamento

In [ ]:
if DST_DIR.exists():
    shutil.rmtree(DST_DIR)
    print("Pasta anterior removida.")

for split in splits:
    for cls in classes:
        (DST_DIR / split / cls).mkdir(parents=True, exist_ok=True)

for split in splits:
    for cls in classes:
        src_folder = SRC_DIR / split / cls
        dst_folder = DST_DIR / split / cls
        images = sorted(src_folder.glob("*.jpg"))
        for img_path in tqdm(images, desc=f"{split}/{cls}"):
            img = Image.open(img_path).convert("RGB")
            img_proc = preprocess(img, METHOD, VALUE)
            img_proc.save(dst_folder / img_path.name, format="JPEG", quality=95)

meta_path = DST_DIR / "metadata.json"
with open(meta_path, "w") as f:
    json.dump({"method": METHOD, "value": VALUE}, f)

print("Concluído.")
print("Metadata:", meta_path)

## 3. Verificação

In [ ]:
import pandas as pd

rows = []
for split in splits:
    for cls in classes:
        count = len(list((DST_DIR / split / cls).glob("*.jpg")))
        rows.append({"split": split, "classe": cls, "imagens": count})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nTotal:", df["imagens"].sum())